About Dataset

Context This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content 5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset. Columns

- asin - ID of the product, like B000FA64PK
- helpful - helpfulness rating of the review - example: 2/3.
- overall - rating of the product.
- reviewText - text of the review (heading).
- reviewTime - time of the review (raw).
- reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
- reviewerName - name of the reviewer.
- summary - summary of the review (description).
- unixReviewTime - unix timestamp.
- Acknowledgements This dataset is taken from Amazon product data, Julian - - - McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/


License to the data files belong to them.

Inspiration-
- Sentiment analysis on reviews.
- Understanding how people rate usefulness of a review/ What factors influence - helpfulness of a review.
- Fake reviews/ outliers.
- Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
- Any other interesting analysis


Best Practises
- Preprocessing And Cleaning - Feature Engineering
- Train Test Split
- BOW,TFIDF,Word2vec
- Train ML algorithms

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

In [2]:
data = pd.read_csv('all_kindle_review.csv')
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [3]:
df = data[['reviewText','rating']]
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [4]:
df.shape

(12000, 2)

In [5]:
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [6]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [7]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

#### Preprocessing and Cleaning - Feature Engineering


In [8]:
# Positive review is 1 and Negative review is 0
df['rating'] = df['rating'].apply(lambda x:0 if x<3 else 1)

In [9]:
df['rating'].unique()
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [10]:
# 1. Lower all the cases 
df['reviewText'] = df['reviewText'].str.lower()

In [11]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [15]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from bs4 import BeautifulSoup

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [17]:
## Removing special characters
df['reviewText']=df['reviewText'].apply(lambda x:re.sub('[^a-z A-z 0-9-]+', '',x))

## Remove the stopswords
df['reviewText']=df['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]))

## Remove url 
df['reviewText']=df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))

## Remove html tags
df['reviewText'] = df['reviewText'].apply(
    lambda x: BeautifulSoup(x, 'html.parser').get_text()
)

## Remove any additional spaces
df['reviewText']=df['reviewText'].apply(lambda x: " ".join(x.split()))

In [18]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [20]:
# Apply lemmatizer 
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [22]:
def lemmatizer_word(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split() ])

In [23]:
df['reviewText']=df['reviewText'].apply(lambda x:lemmatizer_word(x) )

In [24]:
df['reviewText']

0        jace rankin may short he nothing mess man haul...
1        great short read didnt want put read one sitti...
2        ill start saying first four book wasnt expecti...
3        aggie angela lansbury carry pocketbook instead...
4        expect type book library pleased find price right
                               ...                        
11995    valentine cupid vampire- jena ian another vamp...
11996    read seven book series apocalypticadventure on...
11997    book really wasnt cuppa situation man capturin...
11998    tried use charge kindle didnt even register ch...
11999    taking instruction look often hidden world sex...
Name: reviewText, Length: 12000, dtype: object

In [25]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [ ]:
# 2. Train test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df['reviewText'],df['rating'],test_size=0.20) 

In [33]:
from sklearn.feature_extraction.text import CountVectorizer
bow  = CountVectorizer()
X_train_bow = bow.fit_transform(X_train).toarray()
X_test_bow = bow.transform(X_test).toarray()

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf  = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

In [36]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(9600, 35770))

In [37]:
X_train_tfidf

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(9600, 35770))

In [41]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow  = GaussianNB().fit(X_train_bow,y_train)

nb_model_tfidf  = GaussianNB().fit(X_train_tfidf,y_train)


In [42]:
nb_model_bow

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [43]:
nb_model_tfidf

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [44]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report

In [45]:
y_pred_bow = nb_model_bow.predict(X_test_bow)

In [47]:
y_pred_bow

array([1, 1, 1, ..., 0, 1, 0], shape=(2400,))

In [46]:
y_pred_tfidf = nb_model_tfidf.predict(X_test_tfidf)

In [48]:
y_pred_tfidf 

array([1, 1, 1, ..., 0, 1, 0], shape=(2400,))

In [52]:
print("BOW Accuracy_Score: " , accuracy_score(y_test,y_pred_bow))

BOW Accuracy_Score:  0.59125


In [54]:
print(confusion_matrix(y_test,y_pred_bow))

[[540 298]
 [683 879]]


In [53]:
print("TFIDF Accuracy_Score: " , accuracy_score(y_test,y_pred_tfidf))


TFIDF Accuracy_Score:  0.5929166666666666


In [55]:
print(confusion_matrix(y_test,y_pred_tfidf))

[[528 310]
 [667 895]]
